In [ ]:
import os
from pathlib import Path
DATA = Path(os.environ.get('ENSO_DATA_ROOT', Path.cwd() / 'data'))


In [ ]:
from mpl_toolkits.basemap import Basemap
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import itertools
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
import scipy.io as sio
from matplotlib.collections import LineCollection
import matplotlib.patches as mpatches
from scipy.stats import ttest_ind  # 双样本t检验
import cartopy.crs as ccrs  # 地图
import cartopy.feature as cfeature
import colormaps as cmaps  # 确保 colormaps 库已安装
from matplotlib.patches import Rectangle


In [ ]:

# 读数据
data_path_aprc = str(DATA / "mean.mat")
lon = sio.loadmat(data_path_aprc)['lon'].flatten()
lat = sio.loadmat(data_path_aprc)['lat'].flatten()
# idx_lat = np.where((lat >= -30) & (lat <= 30))[0]
# idx_lon = np.where((lon_aprc >= 40) & (lon_aprc <= 300))[0]
apprecession = sio.loadmat(data_path_aprc)['mean_aprc']
landaprc = sio.loadmat(data_path_aprc)['mean_aprc_land24']
# landaprc=landaprc*36.5
ave_tsw_precesion = sio.loadmat(data_path_aprc)['mean_tsw']
lon, lat = np.meshgrid(lon, lat)  # 构建经纬网


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap
import numpy as np

# 配置全局字体样式
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.weight'] = 'normal'
plt.rcParams['font.size'] = 11
plt.rcParams['ytick.direction'] = 'in'

# 设置地图和投影
fig = plt.figure(figsize=(8, 8))
# 修正 ax 的位置参数，归一化坐标应在 [0, 1] 范围内
ax = fig.add_axes([0.1, 0.2, 0.8, 0.7])  # 调整 bottom=0.2 为 colorbar 留出空间

# Mercator 投影
m = Basemap(
    projection='merc',
    llcrnrlat=-61, urcrnrlat=61,
    llcrnrlon=0, urcrnrlon=300,
    resolution='c', ax=ax
)

# 假设 lon, lat, landaprc, ave_tsw_precesion, apprecession, custom_cmap, custom_cmap1 已定义
x, y = m(lon, lat)

# 绘制数据
c3 = m.contourf(x, y, landaprc.T, levels=np.arange(0, 6.01, 0.01), cmap=custom_cmap, extend='both')
c1 = m.contourf(x, y, (ave_tsw_precesion - 273.15).T, levels=np.arange(-2,28, 0.01), cmap=custom_cmap1, extend='both')
h1 = m.contour(x, y, apprecession.T, levels=np.arange(5, 5.01, 0.01), colors='g', linestyles='-', linewidths=1.5)
ax.clabel(h1, inline=True, fontsize=11, fmt='5 mm/day', colors='g')

# 海岸线
m.drawcoastlines(color='gray', linewidth=0.5)

# 经纬线标签，无网格线
m.drawmeridians(np.arange(0, 360, 30), color='none', labels=[True, False, False, True])
m.drawparallels(np.arange(-60, 61, 30), color='none', labels=[True, False, True, False])

# 添加矩形框
rect_lon_start, rect_lat_start = 30, -30
rect_lon_end, rect_lat_end = 240, 30
ax.add_patch(plt.Rectangle(
    m(rect_lon_start, rect_lat_start),
    m(rect_lon_end, rect_lat_end)[0] - m(rect_lon_start, rect_lat_start)[0],
    m(rect_lon_end, rect_lat_end)[1] - m(rect_lon_start, rect_lat_start)[1],
    edgecolor='b', facecolor='none', linewidth=2
))

# 调整 colorbar1
# 自定义 colorbar 轴的位置和大小（更窄，向上靠近）
cax1 = fig.add_axes([0.09, 0.28, 0.6, 0.02])  # [left, bottom, width, height]，width=0.7 更窄
colorbar1 = fig.colorbar(c3, cax=cax1, orientation='horizontal')
colorbar1.set_label('Precipitation [mm/day]', fontsize=11, fontweight='normal')
colorbar1.ax.xaxis.set_tick_params(labelsize=11)
colorbar1.set_ticks(np.arange(0,6.01, 1))
colorbar1.ax.xaxis.set_label_coords(1.25,0.8)  # 微调：x=1.05 靠近尖端，y=0.5 稍微向上
# 调整 colorbar2
cax2 = fig.add_axes([0.09, 0.22, 0.6, 0.02])  # 第二个 colorbar 在第一个下方，宽度相同
colorbar2 = fig.colorbar(c1, cax=cax2, orientation='horizontal')
colorbar2.set_label('SST [°C]', fontsize=11, fontweight='normal')
colorbar2.ax.xaxis.set_label_coords(1.15,0.8)  # 微调：x=1.05 靠近尖端，y=0.5 稍微向上
colorbar2.ax.xaxis.set_tick_params(labelsize=11)
colorbar2.set_ticks(np.arange(-2,28.01,2))

# 保存高分辨率图片
plt.savefig('map_plot_adjusted.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from matplotlib.colors import ListedColormap
# 加载 .rgb 文件中的颜色代码
colorcode = np.loadtxt(str(DATA / "colormaps" / "testcmap (1).rgb"))
colorcode = colorcode /256
# 创建并测试自定义色带
custom_cmap = ListedColormap(colorcode)
data = np.random.rand(8, 8)
plt.imshow(data, cmap=custom_cmap)
plt.colorbar()
plt.show()

In [ ]:
from matplotlib.colors import ListedColormap
# 加载 .rgb 文件中的颜色代码
colorcode = np.loadtxt(str(DATA / "colormaps" / "NCV_bright.rgb"))
colorcode = colorcode /256
# 创建并测试自定义色带
custom_cmap1 = ListedColormap(colorcode)
data = np.random.rand(8, 8)
plt.imshow(data, cmap=custom_cmap1)
plt.colorbar()
plt.show()

In [ ]:
print("landaprc min:", landaprc.min(), "max:", landaprc.max())